# K-Map Practice

Exploring K-maps on binary images and PGS microstructures.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

sys.path.insert(0, '.')
from pgs_tools import make_gaussian_fields, make_lithotype_map

print('Imports OK')

## 1. Helper Functions

Three reusable functions used throughout the notebook.

In [ ]:
def build_kmap(src, tgt, key_fn):
    """Learn a K-map: key_fn(src, r, c) -> pattern tuple, tgt[r,c] -> output.
    Returns dict: pattern -> [count_F0, count_F1]
    """
    kmap = defaultdict(lambda: [0, 0])
    for r in range(src.shape[0]):
        for c in range(src.shape[1]):
            key = key_fn(src, r, c)
            if key is not None:
                kmap[key][int(tgt[r, c])] += 1
    return kmap


def reconstruct(src, kmap, key_fn, border_fill=None):
    """Apply a learned K-map to src. Border pixels use border_fill if provided."""
    out = np.zeros_like(src)
    for r in range(src.shape[0]):
        for c in range(src.shape[1]):
            key = key_fn(src, r, c)
            if key is None:
                out[r, c] = border_fill[r, c] if border_fill is not None else 0
            else:
                c0, c1 = kmap[key]
                out[r, c] = 1 if c1 > c0 else 0
    return out


def accuracy(pred, target, border=1):
    """Pixel accuracy, ignoring the outermost border ring."""
    p, t = pred[border:-border, border:-border], target[border:-border, border:-border]
    m = np.sum(p == t)
    return m, p.size, 100 * m / p.size


# --- Neighbourhood key functions ---

def diag_key(grid, r, c):
    """Centre + 4 diagonal neighbours (5-bit key, 32 possible patterns)."""
    if r == 0 or r == grid.shape[0]-1 or c == 0 or c == grid.shape[1]-1:
        return None
    return (grid[r, c], grid[r-1, c-1], grid[r-1, c+1], grid[r+1, c-1], grid[r+1, c+1])


def full3x3_key(grid, r, c):
    """Full 3x3 neighbourhood (9-bit key, 512 possible patterns)."""
    if r == 0 or r == grid.shape[0]-1 or c == 0 or c == grid.shape[1]-1:
        return None
    return (
        grid[r-1, c-1], grid[r-1, c], grid[r-1, c+1],
        grid[r,   c-1], grid[r,   c], grid[r,   c+1],
        grid[r+1, c-1], grid[r+1, c], grid[r+1, c+1],
    )


print('Helpers loaded.')

## 2. Boolean K-map

Classic 3-variable K-map. Finds the minimal boolean expression for a truth table.

In [ ]:
AB = ['00', '01', '11', '10']
C  = ['0', '1']

F = [
    [1, 1],   # AB=00
    [1, 1],   # AB=01
    [0, 0],   # AB=11
    [0, 0],   # AB=10
]

ones = [ab + c for i, ab in enumerate(AB) for j, c in enumerate(C) if F[i][j] == 1]

expression = []
for idx, name in enumerate(['A', 'B', 'C']):
    bits = [cell[idx] for cell in ones]
    if all(b == '1' for b in bits):   expression.append(name)
    elif all(b == '0' for b in bits): expression.append(name + "'")

print('F =', ' '.join(expression))

# verify
ok = all(
    int(int(ab[0]) == 0) == F[i][j]
    for i, ab in enumerate(AB)
    for j, c in enumerate(C)
)
print('Verification:', '✓ all correct' if ok else '✗ errors found')

## 3. Generate PGS Images

Two binary microstructures from the same Gaussian fields with different threshold splits.
These are used in all sections below.

In [ ]:
field_1, field_2 = make_gaussian_fields(grid_size=200, seed_1=0, seed_2=1)

# image A: 60% Mat1 (black), 40% Mat2 (white)
image_a = (make_lithotype_map(field_1, field_2, Mat1=0.6, Mat2=0.4, Mat3=0.0) > 0).astype(int)
# image B: 40% Mat1 (black), 60% Mat2 (white) — same fields, shifted threshold
image_b = (make_lithotype_map(field_1, field_2, Mat1=0.4, Mat2=0.6, Mat3=0.0) > 0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image_a, cmap='gray'); axes[0].set_title(f'Image A  (white={image_a.mean():.2f})')
axes[1].imshow(image_b, cmap='gray'); axes[1].set_title(f'Image B  (white={image_b.mean():.2f})')
plt.tight_layout()
plt.show()

## 4. Single-Image K-map (Self-consistency)

Build a K-map from image A to itself: diagonal neighbourhood → centre pixel.
Then apply it back to check how well local context predicts each pixel.

In [ ]:
kmap_self = build_kmap(image_a, image_a, diag_key)

print(f'Unique patterns seen: {len(kmap_self)} / 32 possible')
print()
print('(ctr, TL, TR, BL, BR) | F=0    F=1   | majority')
print('-' * 52)
for key in sorted(kmap_self.keys()):
    c0, c1 = kmap_self[key]
    k = tuple(int(x) for x in key)
    print(f'  {k}  | {c0:5d}  {c1:5d} | {1 if c1 > c0 else 0}')

In [ ]:
recon_self = reconstruct(image_a, kmap_self, diag_key, border_fill=image_a)
m, t, pct  = accuracy(recon_self, image_a)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_a,              cmap='gray', vmin=0, vmax=1); axes[0].set_title('Original')
axes[1].imshow(recon_self,           cmap='gray', vmin=0, vmax=1); axes[1].set_title(f'Self-reconstruction ({pct:.1f}%)')
axes[2].imshow(image_a != recon_self, cmap='Reds', vmin=0, vmax=1); axes[2].set_title('Errors (red = wrong)')
plt.tight_layout()
plt.show()
print(f'Accuracy: {m}/{t} ({pct:.1f}%)  — errors are at phase boundaries.')

## 5. Two-Image K-map: A → B

Learn a K-map from image A and apply it to reconstruct image B.
~83% is the expected ceiling — pixels deep inside a phase are easy;
boundary pixels are ambiguous because the same local pattern can map to either value.

In [ ]:
kmap_ab  = build_kmap(image_a, image_b, full3x3_key)
recon_ab = reconstruct(image_a, kmap_ab, full3x3_key, border_fill=image_b)
m, t, pct = accuracy(recon_ab, image_b)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_a,  cmap='gray'); axes[0].set_title('Image A (input)')
axes[1].imshow(image_b,  cmap='gray'); axes[1].set_title('Image B (target)')
axes[2].imshow(recon_ab, cmap='gray'); axes[2].set_title(f'Reconstructed B ({pct:.1f}%)')
plt.tight_layout()
plt.show()
print(f'Accuracy: {m}/{t} ({pct:.1f}%)')

## 6. Difference K-map

Instead of mapping A→B directly, look at what *changed*.
`diff = B - A` marks pixels that flipped: +1 = gained white, -1 = lost white.
Then ask: for each pixel in A, how many white neighbours does it have,
and what fraction of pixels with that count actually flipped?
This is split into gains (black→white) and losses (white→black) to show asymmetry.

In [ ]:
diff = image_b.astype(int) - image_a.astype(int)

print(f'Unchanged:  {np.mean(diff == 0)*100:.1f}%')
print(f'Gained 0→1: {np.mean(diff == 1)*100:.1f}%')
print(f'Lost   1→0: {np.mean(diff ==-1)*100:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_a, cmap='gray'); axes[0].set_title('Image A')
axes[1].imshow(image_b, cmap='gray'); axes[1].set_title('Image B')
im = axes[2].imshow(diff, cmap='RdYlGn', vmin=-1, vmax=1)
axes[2].set_title('Difference  (green=gained, red=lost)')
plt.colorbar(im, ax=axes[2])
plt.tight_layout()
plt.show()

In [ ]:
rows, cols = image_a.shape
gain_by_n = defaultdict(lambda: [0, 0])   # black pixels: [stayed_black, became_white]
loss_by_n = defaultdict(lambda: [0, 0])   # white pixels: [stayed_white, became_black]

for r in range(1, rows - 1):
    for c in range(1, cols - 1):
        n_white = int(
            image_a[r-1,c-1] + image_a[r-1,c] + image_a[r-1,c+1] +
            image_a[r,  c-1]                   + image_a[r,  c+1] +
            image_a[r+1,c-1] + image_a[r+1,c]  + image_a[r+1,c+1]
        )
        if image_a[r, c] == 0:
            gain_by_n[n_white][int(diff[r, c] == 1)] += 1
        else:
            loss_by_n[n_white][int(diff[r, c] == -1)] += 1

print(f'Total gained: {int(np.sum(diff==1))}   Total lost: {int(np.sum(diff==-1))}')
print()
print('Nbrs | Gain prob (black→white)        | Loss prob (white→black)')
print('-' * 68)
for n in range(9):
    gs, gf = gain_by_n[n];  g = gf/(gs+gf) if (gs+gf) > 0 else None
    ls, lf = loss_by_n[n];  l = lf/(ls+lf) if (ls+lf) > 0 else None
    g_str = f'{g:.3f} {"█"*int(g*15)}' if g is not None else '  —'
    l_str = f'{l:.3f} {"█"*int(l*15)}' if l is not None else '  —'
    if (gs+gf) > 0 or (ls+lf) > 0:
        print(f'  {n}  | {g_str:<32s} | {l_str}')

## 7. Probabilistic K-map

Instead of a majority vote, output the probability p = count_1 / (count_0 + count_1).
The result is a greyscale confidence map — pixels near 0 or 1 are certain,
pixels near 0.5 are ambiguous boundary pixels.

In [ ]:
# build probabilistic K-map
counts = defaultdict(lambda: [0, 0])
for r in range(image_a.shape[0]):
    for c in range(image_a.shape[1]):
        key = full3x3_key(image_a, r, c)
        if key is not None:
            counts[key][int(image_b[r, c])] += 1

prob_kmap = {k: v[1] / (v[0] + v[1]) for k, v in counts.items()}

# apply to get probability map
prob_map = np.zeros(image_a.shape, dtype=float)
for r in range(image_a.shape[0]):
    for c in range(image_a.shape[1]):
        key = full3x3_key(image_a, r, c)
        prob_map[r, c] = prob_kmap.get(key, float(image_b[r, c]))

uncertain = np.sum((prob_map > 0.3) & (prob_map < 0.7))
print(f'Confident pixels (p < 0.3 or p > 0.7): {prob_map.size - uncertain}  ({100*(prob_map.size-uncertain)/prob_map.size:.1f}%)')
print(f'Uncertain pixels (0.3 < p < 0.7):       {uncertain}  ({100*uncertain/prob_map.size:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_a,  cmap='gray', vmin=0, vmax=1); axes[0].set_title('Image A (input)')
axes[1].imshow(image_b,  cmap='gray', vmin=0, vmax=1); axes[1].set_title('Image B (target)')
im = axes[2].imshow(prob_map, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Probability map  (grey = uncertain)')
plt.colorbar(im, ax=axes[2])
plt.tight_layout()
plt.show()

## Summary

**What we found:**

- **Self-consistency (Section 4):** ~100% accuracy. The K-map perfectly recovers image A from itself because every pattern has one consistent answer.
- **A→B reconstruction (Section 5):** ~83% accuracy. Interior pixels are easy; boundary pixels are ambiguous because the same local pattern appears on both sides of the shifted threshold.
- **Difference K-map (Section 6):** flip probability is almost entirely determined by neighbour count. Pixels with 1–3 white neighbours flip with near certainty; interior pixels rarely flip.
- **Probabilistic K-map (Section 7):** ~57% of pixels are uncertain (p between 0.3 and 0.7) — these are the boundary pixels the binary K-map gets wrong.

**Why reconstruction fails for PGS threshold shifts:**
Whether a boundary pixel flips depends on its underlying continuous Gaussian value, which the binary K-map cannot see. This is not a bug — it is the fundamental limit of local binary rules for global threshold changes.

**Useful applications going forward:**
- K-map fingerprint (pattern frequency distribution) as a distance metric between microstructures — inverse problem direction.
- Difference K-map to characterise what local conditions drive phase changes under a CA rule.